# 04 · Tools and the ReAct Agent

> **Kernel:** `Phase 5 · LangGraph` · **Needs:** Ollama (a model with tool calling;
> the default `qwen3-coder:30b` works).

A model on its own can only produce text. **Tools** let it act: look things up,
compute, call APIs. In this notebook you'll do one tool-call round trip by
hand, then build the classic **ReAct agent** (reason → act → observe → repeat)
as a LangGraph graph. Finally you'll meet `create_agent`, the prebuilt version,
and see exactly what it is.

**You will learn**
- How tool calling actually works: the model *requests*, your code *executes*
- Defining tools with `@tool`: the docstring and type hints become the schema
- One complete tool round trip by hand: `bind_tools`, `tool_calls`, `ToolMessage`
- The ReAct loop as a graph: model node + `ToolNode` + `tools_condition`
- Tool errors: why the default **crashes your graph**, and how to make the agent recover
- Stopping runaway loops with `recursion_limit`
- `create_agent`: the prebuilt agent, its middleware, and when to build your own

**Prerequisites:** notebooks 01–03.

In [ ]:
import sys, pathlib
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 01_LANGGRAPH folder
sys.path.insert(0, str(SECTION))

import time
from typing import Annotated
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from common import get_model

model = get_model()

## 1 · Why tools, and what "tool calling" really means

Models don't know today's data, can't do reliable arithmetic, and can't touch
your systems. Tools fix that, but it's important to be precise about **who does
what**:

```
 your code                                   the model
 ─────────                                   ─────────
 1. send messages + TOOL SCHEMAS  ─────────►  reads the schemas (name, description, args)
                                  ◄─────────  2. replies with an AIMessage containing
                                                 tool_calls = [{name, args, id}]   ← a REQUEST
 3. YOUR CODE runs the function
 4. send back a ToolMessage       ─────────►  reads the result
    (content, tool_call_id)       ◄─────────  5. answers, or requests more tools
```

The model **never executes anything**. It only produces a structured request.
Your program decides whether and how to run it. That's where safety checks,
approvals (notebook 07) and error handling live.

## 2 · Defining tools

`@tool` turns a Python function into a tool. The **function name** becomes the
tool name, the **docstring** its description, and the **type hints** its
argument schema. That's what the model reads, so write them for the model:

In [ ]:
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city. Use for any question about weather."""
    time.sleep(1.0)                                  # pretend: a slow API
    fake = {'pune': '31°C, humid', 'oslo': '4°C, rain', 'lima': '19°C, cloudy'}
    return f"{city}: {fake.get(city.lower(), '20°C, clear')}"


@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert an amount between currencies (EUR, USD, INR) using today's rate."""
    rates = {'EUR': 1.0, 'USD': 1.08, 'INR': 90.5}
    value = amount / rates[from_currency.upper()] * rates[to_currency.upper()]
    return f'{amount} {from_currency} = {value:.2f} {to_currency}'


print('name       :', convert_currency.name)
print('description:', convert_currency.description)
print('schema     :', convert_currency.tool_call_schema.model_json_schema()['properties'])

**Tool design matters more than prompt tweaking.** The model picks tools by
reading names and descriptions, and fills arguments from the types:

| Do | Why |
|---|---|
| Name tools as clear verbs: `get_weather`, `search_documents` | the name is the model's first hint |
| Say **when** to use it in the docstring | helps the model choose between similar tools |
| Use precise types (`float`, `Literal['EUR','USD']`) | the schema constrains the arguments |
| Return short, informative strings | the result is fed back into the context window |
| Keep each tool to one job | small tools compose; big ones confuse |

## 3 · One round trip, by hand

Before any graph, let's do the protocol manually. `bind_tools` attaches the
schemas to every call the model makes:

In [ ]:
model_with_tools = model.bind_tools([get_weather, convert_currency])

messages = [HumanMessage('What is the weather in Pune and in Oslo?')]
ai = model_with_tools.invoke(messages)                # step 2: the model REQUESTS tools
print('content   :', repr(ai.content))
for call in ai.tool_calls:
    print('tool_call :', call)

The model returned no answer, only **two requests** (one per city), each with
an `id`. Now we play the role of the program: run each tool and send back a
`ToolMessage` carrying the **same `tool_call_id`**, so the model knows which
result answers which request.

In [ ]:
tools_by_name = {'get_weather': get_weather, 'convert_currency': convert_currency}

messages.append(ai)                                   # the model's request is part of history
for call in ai.tool_calls:                            # step 3: WE execute
    result = tools_by_name[call['name']].invoke(call['args'])
    messages.append(ToolMessage(content=result, tool_call_id=call['id']))

final = model_with_tools.invoke(messages)             # step 5: the model reads results and answers
print(final.content)

That's the whole protocol. An **agent** is just this in a loop: keep calling the
model and running whatever it requests, until it replies with no `tool_calls`.

## 4 · The ReAct agent as a graph

ReAct (*Reason + Act*) is that loop. As a graph it's two nodes and one
conditional edge:

```
 START ──► agent ──(has tool_calls?)──yes──► tools ──┐
             ▲            │                          │
             │            no                         │
             │            ▼                          │
             │           END                         │
             └───────────────────────────────────────┘
```

LangGraph ships the two pieces you need:
- **`ToolNode(tools)`**: a node that runs every tool call in the last
  `AIMessage` (in parallel) and returns the `ToolMessage`s.
- **`tools_condition`**: a router that returns `'tools'` if the last message has
  tool calls, else `END`.

In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition

tools = [get_weather, convert_currency]
llm = model.bind_tools(tools)


def agent(state: MessagesState) -> dict:
    return {'messages': [llm.invoke(state['messages'])]}


builder = StateGraph(MessagesState)
builder.add_node('agent', agent)
builder.add_node('tools', ToolNode(tools))
builder.add_edge(START, 'agent')
builder.add_conditional_edges('agent', tools_condition)     # → 'tools' or END
builder.add_edge('tools', 'agent')                           # ← the loop
react = builder.compile()

print(react.get_graph().draw_ascii())

In [ ]:
def show_run(graph, question, **kwargs):
    t0 = time.perf_counter()
    for update in graph.stream({'messages': [('user', question)]}, stream_mode='updates', **kwargs):
        for node, delta in update.items():
            for msg in (delta or {}).get('messages', []):     # some steps (e.g. middleware) update nothing
                calls = [f"{c['name']}({c['args']})" for c in getattr(msg, 'tool_calls', [])]
                text = (msg.content or '')[:110].replace('\n', ' ')
                print(f'{time.perf_counter() - t0:5.1f}s [{node:<5}] {msg.type:<4} {text} {" ".join(calls)}')

show_run(react, 'Weather in Pune, Oslo and Lima? Also convert 250 EUR to INR.')

Read the timeline:

1. The model requested **four tools in one turn** (three weathers and a conversion).
2. `ToolNode` ran them **concurrently**. The three 1-second weather calls
   finished together, not after 3 seconds.
3. The model read all four results and answered. There were no tool calls
   left, so `tools_condition` routed to `END`.

## 5 · When a tool fails

Real tools fail: bad input, network errors, missing records. Let's add a tool
that can raise:

In [ ]:
@tool
def divide(a: float, b: float) -> float:
    """Divide a by b."""
    return a / b


def build_react(tool_node):
    llm = model.bind_tools([divide])
    b = StateGraph(MessagesState)
    b.add_node('agent', lambda s: {'messages': [llm.invoke(s['messages'])]})
    b.add_node('tools', tool_node)
    b.add_edge(START, 'agent')
    b.add_conditional_edges('agent', tools_condition)
    b.add_edge('tools', 'agent')
    return b.compile()


QUESTION = 'Use the divide tool to compute 10 / 0, then explain what happened.'
try:
    build_react(ToolNode([divide])).invoke({'messages': [('user', QUESTION)]})
except ZeroDivisionError as err:
    print('💥 the whole graph crashed:', repr(err))

**By default, an exception inside a tool propagates and kills the run.** That's
a deliberate choice: a bug in your tool shouldn't be silently hidden from you.
But for an agent it's often wrong, because the model could have recovered by
trying different arguments or explaining the problem to the user.

`handle_tool_errors=True` turns the exception into an error `ToolMessage` that
the model reads:

In [ ]:
result = build_react(ToolNode([divide], handle_tool_errors=True)).invoke({'messages': [('user', QUESTION)]})
for m in result['messages']:
    status = f" [status={m.status}]" if m.type == 'tool' else ''
    print(f'{m.type:<5}{status} {(m.content or "(tool call)")[:150]}')

Now the model sees `Error: ZeroDivisionError(...)` as an observation and
explains it. There are two kinds of failures:

| Failure | Example | Default behaviour |
|---|---|---|
| **Invalid arguments** (schema validation) | `divide(a='ten', b=2)` | already returned to the model as an error `ToolMessage` |
| **Exception while executing** | `ZeroDivisionError`, a timeout, an API error | **raised**: the graph stops |

`handle_tool_errors` also accepts an exception type (or tuple of types) to
catch only those, or a function `(exception) -> str` to write your own message.
Catching only *expected* errors (like "record not found") and letting real
bugs crash is usually the best of both worlds.

## 6 · Stopping runaway loops

An agent is a cycle, and a cycle needs a guard. Suppose a model keeps
requesting a tool forever (it happens: a status that's always "pending", a
search that never satisfies it). We'll simulate that model with a plain function
so it happens every time:

In [ ]:
from langgraph.errors import GraphRecursionError

@tool
def check_status(job_id: str) -> str:
    """Check whether a background job has finished."""
    return 'still running, check again later'


def stubborn_model(state: MessagesState) -> dict:
    n = len(state['messages'])
    return {'messages': [AIMessage('', tool_calls=[
        {'name': 'check_status', 'args': {'job_id': 'job-42'}, 'id': f'call-{n}'}])]}


b = StateGraph(MessagesState)
b.add_node('agent', stubborn_model)
b.add_node('tools', ToolNode([check_status]))
b.add_edge(START, 'agent')
b.add_conditional_edges('agent', tools_condition)
b.add_edge('tools', 'agent')
looping = b.compile()

try:
    looping.invoke({'messages': [('user', 'Is job-42 done?')]}, config={'recursion_limit': 8})
except GraphRecursionError as err:
    print('stopped:', str(err).splitlines()[0])

`recursion_limit` is the maximum number of **supersteps** in one run (default
**25**; each `agent` or `tools` visit is one). When it's hit, LangGraph raises
`GraphRecursionError` rather than looping forever. Set it deliberately. A simple
agent needs `2 × (expected tool rounds) + 1`, plus some margin.

Hitting the limit is a crash, though. For a *graceful* stop (e.g. "answer with
what you have after 5 tool rounds"), count rounds in state and route to a final
node, as you'll do in notebook 05.

## 7 · `create_agent`: the prebuilt agent

Everything in §4–§6 is so common that LangChain ships it as one function.
`create_agent` returns a compiled LangGraph graph, the same loop you just built:

In [ ]:
from langchain.agents import create_agent

prebuilt = create_agent(
    model,
    tools=[get_weather, convert_currency],
    system_prompt='You are a travel assistant. Use tools for facts; be concise.',
)
print(type(prebuilt).__name__)
print(prebuilt.get_graph().draw_ascii())
show_run(prebuilt, 'Is it warmer in Pune or Lima right now? And what is 100 USD in INR?')

Same shape (`model ⇄ tools`), same runtime, same streaming. Because it's a
normal compiled graph, you can use it **as a node inside your own graph**,
which is how agents get embedded in workflows (notebooks 06 and 10).

`create_agent` customises behaviour with **middleware**: hooks that run
around model and tool calls. Two useful ones are a readable fix for §5 and a
graceful version of §6:

In [ ]:
from langchain.agents.middleware import ToolErrorMiddleware, ModelCallLimitMiddleware


def explain_tool_error(exc, request):
    # Return text for the model, or None to let the exception propagate.
    if isinstance(exc, ArithmeticError):
        return f"`{request.tool_call['name']}` failed: {exc}. Check the inputs."
    return None                                    # anything unexpected still crashes


careful = create_agent(
    model,
    tools=[divide],
    middleware=[
        ToolErrorMiddleware(explain_tool_error),   # expected errors → message for the model
        ModelCallLimitMiddleware(run_limit=4),     # at most 4 model calls per run, then stop
    ],
)
show_run(careful, 'Use divide to compute 10 / 0, then explain what happened.')

Other built-in middleware includes human approval for tool calls, summarising
long histories, PII redaction, model fallbacks, and retrying tools. You'll use
the human-approval one in notebook 07.

**Prebuilt or hand-built?**

| Use `create_agent` when… | Build the graph yourself when… |
|---|---|
| you want a standard tool-calling agent | you need custom steps (validation, routing, parallel branches) |
| the behaviour you need exists as middleware | the loop itself must change shape |
| it's one node inside a bigger workflow | you're teaching or debugging the mechanics |

## 8 · Trade-offs of the ReAct pattern

- **Flexible but unpredictable.** The model decides how many rounds, which tools
  and in what order. Great for open-ended tasks, poor for guaranteed procedures
  (put those in explicit edges).
- **Cost grows every round.** Each round re-sends the whole conversation,
  including all tool results. Keep results short and cap the rounds.
- **Tool quality dominates.** A vague tool description causes more failures than
  any prompt can fix.
- **Parallel tool calls are free speed**, if your tools are independent and
  thread-safe. `ToolNode` runs them concurrently.

## 9 · Common mistakes

- **Letting tool exceptions crash the agent** when they're expected. Use
  `handle_tool_errors` / `ToolErrorMiddleware`, but only for errors the model
  can act on.
- **Returning huge tool outputs** (whole files, full API responses). They flood
  the context. Return what the model needs.
- **Forgetting the `tool_call_id`** when writing `ToolMessage`s by hand. The
  model can't match results to requests.
- **No loop guard.** Always set `recursion_limit` or a call-limit middleware.
- **Side-effecting tools without approval.** A model can call `delete_file` as
  easily as `read_file`. Gate risky tools (notebook 07).

## 10 · Check yourself

<details><summary><b>1.</b> When a model "calls a tool", what does it actually produce, and who runs the function?</summary>

It produces an `AIMessage` with `tool_calls`: structured requests with a name, arguments and an id. Your program (e.g. `ToolNode`) runs the function and sends back a `ToolMessage` with the matching `tool_call_id`. The model never executes code.
</details>

<details><summary><b>2.</b> Where does a <code>@tool</code>'s schema come from, and why does that matter?</summary>

From the function name (tool name), the docstring (description) and the type hints (arguments). It's all the model sees when deciding which tool to call and how, so clear names, "when to use" docstrings and precise types directly improve reliability.
</details>

<details><summary><b>3.</b> Draw the ReAct graph and name the prebuilt pieces.</summary>

`START → agent`, then a conditional edge from `agent` (`tools_condition`) to `tools` if the last message has tool calls, else `END`, plus `tools → agent`. The pieces are `ToolNode` (runs the calls in parallel) and `tools_condition` (the router).
</details>

<details><summary><b>4.</b> A tool raises <code>ZeroDivisionError</code>. What happens by default, how do you change it, and should you always change it?</summary>

By default the exception propagates and stops the graph. `ToolNode(..., handle_tool_errors=True)` (or `ToolErrorMiddleware` in `create_agent`) turns it into an error `ToolMessage` the model can react to. Not always: catch expected, recoverable errors, and let genuine bugs crash so you notice them.
</details>

<details><summary><b>5.</b> What does <code>recursion_limit</code> count, and what's a graceful alternative to hitting it?</summary>

Supersteps in one run (default 25). Hitting it raises `GraphRecursionError`. For a graceful stop, count rounds in state and route to a "finish with what you have" node, or use `ModelCallLimitMiddleware` in `create_agent`.
</details>

<details><summary><b>6.</b> What is <code>create_agent</code>, in LangGraph terms?</summary>

A function that builds and compiles the standard model ⇄ tools loop as a LangGraph graph, with middleware hooks. Its result is a normal compiled graph: you can invoke, stream, checkpoint it, or use it as a node in a larger graph.
</details>

## Takeaway

Tool calling is a protocol: the model **requests** (`tool_calls`), your code
**executes** and replies with `ToolMessage`s. The ReAct agent is that exchange in
a loop: `agent ⇄ tools`, routed by `tools_condition`, run by `ToolNode`, and
packaged as `create_agent`. Make expected tool errors visible to the model,
always guard the loop, and gate tools with side effects.

Next → `05_control_flow.ipynb`: LLM routers, guarded cycles, parallel fan-out,
map-reduce with `Send`, and `Command` for "update and go to".